In [1]:
from elmo_data import Batcher
import torch
from torch import nn
import os, random
import numpy as np

import pandas as pd
from sklearn.model_selection import train_test_split
from nltk.tokenize import TreebankWordTokenizer
from datasets import load_dataset

def reset_random_seeds():
    os.environ['PYTHONHASHSEED']=str(2)
    torch.manual_seed(2)
    np.random.seed(2)
    random.seed(2)
    torch.cuda.manual_seed(2)
    torch.cuda.manual_seed_all(2)

In [2]:
import re
from string import punctuation
from tqdm.auto import tqdm
import contractions as con

def preprocess_text(x):

    x = str(x).replace('&amp;','and').replace('&quot;','').lower()
    x = re.sub(r'@[A-Za-z0-9_]{1,15}', 'USER', x)
    x = con.fix(x)
    x = re.sub(r'&#x[0-9A-Fa-f]+;','',x)
    x = re.sub(r'&#\d+;',"'",x)
    x = re.sub(r'[^\x00-\x7F]+', "'",x)
    
    url_pattern = r'http\S+|www\S+'
    x = re.sub(url_pattern, 'LINK', x)
    
    punct_to_keep = """!,.:#?"-;//%$(){}@^*+<=>\\|'"""
    punct = ''.join([p for p in punctuation if p not in punct_to_keep])
    trans = str.maketrans(punct, ' ' * len(punct))
    x = x.translate(trans)
    x = ''.join(x)
    x = re.sub(r'([!"#$%&\()*+,-./:;<=>?@\\^_`{|}~])\s*\1+', r'\1', x)
    x = re.sub(r'([!"#$%&\()*+,-./:;<=>?@\\^_`{|}~])', r' \1 ', x)
    x = re.sub(r'\s+', ' ', x).strip().replace("'s "," 's ")
    x = x.replace("\\'"," '").replace("'"," ' ")
    
    return re.sub(r'\s+', ' ', x).strip()

In [3]:
class SelfAttention(nn.Module):

    def __init__(self,d_model,n_heads,look_ahead_mask=False):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.attn_scores = None
        self.d_head = self.d_model // self.n_heads
        self.look_ahead_mask = look_ahead_mask

        self.q = nn.Linear(self.d_model,self.d_model,False)
        self.k = nn.Linear(self.d_model, self.d_model, False)
        self.v = nn.Linear(self.d_model, self.d_model, False)

        self.attn_drop = nn.Dropout(0.1)
        self.lin = nn.Linear(d_model,d_model)
        self.lin_drop = nn.Dropout(0.1)

    def _split_heads(self,x):
        x = torch.reshape(x,shape=(x.shape[0],x.shape[1],self.n_heads,self.d_head))
        return torch.permute(x,(0,2,1,3))

    def forward(self,x):

        x,mask = x
        q,k,v = x
        mask = mask[:,torch.newaxis,torch.newaxis,:]
        maxlen = q.shape[1]
        b = q.shape[0]

        q = self.q(q)
        k = self.k(k)
        v = self.v(v)

        q = self._split_heads(q)
        k = self._split_heads(k)
        v = self._split_heads(v)

        k_trans = torch.permute(k,(0,1,3,2))
        attn = torch.matmul(q,k_trans) / (self.d_head ** 0.5)
        attn = torch.where(mask,-1e8,attn)

        if self.look_ahead_mask:
            look_ahead_mask = torch.triu(torch.ones(maxlen, maxlen, device=attn.device),
                                         diagonal=1).bool()
            attn = attn.masked_fill(look_ahead_mask, -1e8)

        attn = torch.softmax(attn,-1)
        self.attn_scores = attn
        attn = self.attn_drop(attn)

        x = torch.matmul(attn,v)
        x = x.permute(0, 2, 1, 3)
        x = torch.reshape(x,(b,maxlen,self.d_model))
        return x


In [4]:
from collections import Counter

class WordTokenizer:

    def __init__(self, vocab_size=10000):

        self.vocab_size = vocab_size

        self.pad_token = "<pad>"
        self.unk_token = "<unk>"
        self.sos_token = "<sos>"
        self.eos_token = "<eos>"

        self.special_tokens = [
            self.pad_token,
            self.unk_token,
            self.sos_token,
            self.eos_token
        ]

        self.word2id = {}
        self.id2word = {}

    def fit(self, texts):

        counter = Counter()

        for text in texts:
            tokens = text.split()
            counter.update(tokens)

        most_common = counter.most_common(
            self.vocab_size - len(self.special_tokens)
        )

        vocab_words = [w for w, _ in most_common]

        vocab = self.special_tokens + vocab_words

        self.word2id = {w: i for i, w in enumerate(vocab)}
        self.id2word = {i: w for w, i in self.word2id.items()}

    def encode(self, text, add_special_tokens=True):

        tokens = text.split()

        ids = []

        if add_special_tokens:
            ids.append(self.word2id[self.sos_token])

        for token in tokens:
            ids.append(
                self.word2id.get(token, self.word2id[self.unk_token])
            )

        if add_special_tokens:
            ids.append(self.word2id[self.eos_token])

        return ids

    def decode(self, ids, remove_special=True):

        words = []

        for i in ids:
            w = self.id2word.get(i, self.unk_token)

            if remove_special and w in self.special_tokens:
                continue

            words.append(w)

        return " ".join(words)

    def pad(self, sequences, max_len=None):

        if max_len is None:
            max_len = max(len(s) for s in sequences)

        padded = []

        for seq in sequences:

            seq = seq[:max_len]

            padding = [self.word2id[self.pad_token]] * (max_len - len(seq))

            padded.append(seq + padding)

        return padded

    def encode_batch(self, texts, pad=True, max_len=None):

        sequences = [self.encode(t) for t in texts]

        if pad:
            sequences = self.pad(sequences, max_len)

        return sequences

In [5]:
class AttentionPool(nn.Module):
    
    def __init__(self,dim):
        super().__init__()
        self.dim = dim 
        self.score = None
        self.w = nn.Linear(dim,1,bias=False)
        
    def forward(self,x,mask=None):
        attn = self.w(x)
        if mask is not None:
            mask = mask[:,:,torch.newaxis]
            attn = torch.where(mask,-1e8,attn)
        attn = torch.softmax(attn,axis=1)
        self.score = attn
        x = x * attn
        return x.sum(1)

In [6]:
def load_glove_model(path, dim=50):
    glove_model = {}

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.rstrip().split()
            
            if len(parts) < dim + 1:
                continue

            word = parts[0]
            vector = parts[-dim:] 

            try:
                embedding = np.array(vector, dtype=np.float32)
                glove_model[word] = embedding
            except ValueError:
                continue

    print(f"{len(glove_model)} words loaded!")
    return glove_model

In [7]:
file = "/media/bibek/New Volume/dolma_300_2024_1.2M.100_combined.txt"

emb = load_glove_model(file,dim=300)

1200001 words loaded!


In [8]:
def initialize_embeddings(vocab_size,embed_dim = 300):

    limit = np.sqrt(6 / (embed_dim + embed_dim))
    w_emb = np.random.uniform(-limit, limit, (vocab_size, embed_dim))

    for w,idx in tok.word2id.items():
        try: w_emb[idx] = emb[w]
        except: pass
        
    w_emb = torch.tensor(w_emb,dtype=torch.float32)
        
    return w_emb
    

In [9]:
from torchinfo import summary
from elmo_layers import ELMoLayer


class ELMoClassifier(nn.Module):
    
    def __init__(self,vocab,glove_emb,dim=128,elmo_dim=256,emb_dim=300):
        
        super().__init__()
        self.dim = dim
        self.vocab = vocab
        self.emb = nn.Embedding(vocab,emb_dim,_weight=glove_emb)
        self.emb.requires_grad_(requires_grad=False)
        self.rnn = nn.GRU(emb_dim + elmo_dim,dim,
                          batch_first=True)
        self.out = nn.Linear(dim,n_classes) 
        self.elmo = ELMoLayer()
        self.elmo.load_state_dict(torch.load("elmo_layer.pth"))
        self.pool = AttentionPool(dim)
        self.avg = nn.AdaptiveAvgPool1d(1)
        self.attn = SelfAttention(128,1)
        
    def forward(self,x,x_,test=False):
        mask = x == 0
        elmo_x = self.elmo(x_)
        x = self.emb(x)
        x = torch.concat([x,elmo_x],dim=-1)
        rnn,_ = self.rnn(x) 
        
        x = self.pool(rnn,mask=mask)
        
        if test:
            return rnn
        
        return self.out(x)


In [10]:

class AttnClassifier(nn.Module):
    
    def __init__(self,vocab,glove_emb,dim=128,emb_dim=300):
        
        super().__init__()
        self.dim = dim
        self.vocab = vocab
        self.emb = nn.Embedding(vocab,emb_dim,_weight=glove_emb)
        self.emb.requires_grad_(requires_grad=False)
        self.rnn = nn.GRU(emb_dim,dim,batch_first=True)
        self.out = nn.Linear(dim,n_classes) 
        self.pool = AttentionPool(dim)
        self.attn = SelfAttention(128,1)
        
    def forward(self,x,test=False):
        mask = x == 0
        x = self.emb(x)
        rnn,_ = self.rnn(x) 
        rnn = self.attn([[rnn,rnn,rnn],mask])
        x = self.pool(rnn,mask=mask)
        
        if test:
            return rnn
        
        return self.out(x)


In [11]:
class BaseClassifier(nn.Module):
    
    def __init__(self,vocab,glove_emb,dim=128,emb_dim=300):
        
        super().__init__()
        self.dim = dim
        self.vocab = vocab
        self.emb = nn.Embedding(vocab,emb_dim,_weight=glove_emb)
        self.emb.requires_grad_(requires_grad=False)
        self.rnn = nn.GRU(emb_dim,dim,batch_first=True)
        self.out = nn.Linear(dim,n_classes) 
        self.pool = AttentionPool(dim)
        
    def forward(self,x,test=False):
        mask = x == 0
        x = self.emb(x)
        rnn,_ = self.rnn(x) 
        x = self.pool(rnn,mask=mask)
        
        if test:
            return rnn
        
        return self.out(x)


In [12]:
import numpy as np


tok = WordTokenizer(10000)

def prepare_data(train_data,test_data,val_data,tok,elmo=False):

    xtrain,ytrain = train_data
    xtest,ytest = test_data
    xval,yval = val_data
    
    if elmo:

        elmo_train = [tree_bank.tokenize(x) for x in xtrain]
        elmo_test = [tree_bank.tokenize(x) for x in xtest]
        elmo_val = [tree_bank.tokenize(x) for x in xval]

        elmo_vocab_file = "vocab-2016-09-10.txt"
        batcher = Batcher(elmo_vocab_file,20)
        elmo_train = batcher.batch_sentences(elmo_train)
        elmo_test = batcher.batch_sentences(elmo_test)
        elmo_val = batcher.batch_sentences(elmo_val)

    tok.fit(xtrain)
    xtrain = tok.encode_batch(xtrain)
    xtest = tok.encode_batch(xtest)
    xval = tok.encode_batch(xval)

    xtrain = np.array(xtrain)
    xtest = np.array(xtest)
    xval = np.array(xval)

    ytest = ytest.astype(np.int32)
    ytrain = ytrain.astype(np.int32)
    yval = yval.astype(np.int32)

    vocab = len(tok.word2id)
    
    data = {}
    
    if elmo:
        
        data['train'] = (xtrain,elmo_train,ytrain)
        data['test'] = (xtest,elmo_test,ytest)
        data['val'] = (xval,elmo_val,yval)
    else:
        data['train'] = (xtrain,ytrain)
        data['test'] = (xtest,ytest)
        data['val'] = (xval,yval)
    
    return vocab,data


In [13]:
from tqdm.auto import tqdm

def train_elmo_model(data,model,opt,loss_fn,epochs=7):
    
    train,valid = data

    losses = {'train':[],'valid':[]}

    for e in range(1,epochs + 1):

        print(f"{e}/{epochs}")

        loss = 0
        model.train()

        for i,batch in enumerate(tqdm(train),1):

            lr = lr_scheduler[i]
            opt.param_groups[0]['lr'] = lr

            opt.zero_grad()

            x,z,y = batch
            if n_classes == 1:
                y = y.float().unsqueeze(1)
            elif n_classes > 1:
                y = y.long()
            pred = model(x,z)
            batch_loss = loss_fn(pred,y)
            batch_loss.backward()
            opt.step()

            loss += batch_loss.item()

        loss = round(loss / len(train),4)
        losses['train'].append(loss)

        print(f'Train Loss = {loss}')

        model.eval()
        loss = 0

        with torch.no_grad():

            for i,batch in enumerate(valid):

                x,z,y = batch
                if n_classes == 1:
                    y = y.float().unsqueeze(1)
                elif n_classes > 1:
                    y = y.long()
                pred = model(x,z)
                batch_loss = loss_fn(pred,y)
                loss += batch_loss.item()

        loss = round(loss / len(test),4)
        print(f"Val Loss = {loss}")


        if e == 1 or loss < min(losses['valid']):
            torch.save(model.state_dict(),'saved_weights.pt')
            print("--weights saved")

        losses['valid'].append(loss)

        print()

In [14]:
def train_model(data,model,opt,loss_fn,epochs=7):
    
    train,valid = data

    losses = {'train':[],'valid':[]}

    for e in range(1,epochs + 1):

        print(f"{e}/{epochs}")

        loss = 0
        model.train()

        for i,batch in enumerate(tqdm(train),1):

            lr = lr_scheduler[i]
            opt.param_groups[0]['lr'] = lr

            opt.zero_grad()

            x,y = batch
            if n_classes == 1:
                y = y.float().unsqueeze(1)
            elif n_classes > 1:
                y = y.long()
            pred = model(x)
            batch_loss = loss_fn(pred,y)
            batch_loss.backward()
            opt.step()

            loss += batch_loss.item()

        loss = round(loss / len(train),4)
        losses['train'].append(loss)

        print(f'Train Loss = {loss}')

        model.eval()
        loss = 0

        with torch.no_grad():

            for i,batch in enumerate(valid):

                x,y = batch
                if n_classes == 1:
                    y = y.float().unsqueeze(1)
                elif n_classes > 1:
                    y = y.long()
                pred = model(x)
                batch_loss = loss_fn(pred,y)
                loss += batch_loss.item()

        loss = round(loss / len(test),4)
        print(f"Val Loss = {loss}")


        if e == 1 or loss < min(losses['valid']):
            torch.save(model.state_dict(),'saved_weights.pt')
            print("--weights saved")

        losses['valid'].append(loss)

        print()

In [43]:
sents = """
The movie banks good money
The movie starts by the river banks
The movie was shot well
He was shot in the movie
The film was a hit!
It was an amazing film
"""

sents = sents.split('\n')[1:-1]

word_1 = "banks"
word_2 = "shot"
word_3 = "film"

def polysemy_test(word,sent_pairs,model,elmo=False):
    
    sent1,sent2 = sent_pairs
    
    w1_id = sent1.split().index(word) + 1
    w2_id = sent2.split().index(word) + 1
    
    sent1 = preprocess_text(sent1)
    sent2 = preprocess_text(sent2)
    inp1 = tok.encode(sent1)
    inp2 = tok.encode(sent2)
    padding_1 = np.zeros((1,128 - len(inp1),20))
    padding_2 = np.zeros((1,128 - len(inp2),20))
    

    inp1 = inp1 + [0] * (128 - len(inp1))
    inp2 = inp2 + [0] * (128 - len(inp2))
    inp1 = torch.tensor(np.array(inp1)[np.newaxis,:],device=device,dtype=torch.int32)
    inp2 = torch.tensor(np.array(inp2)[np.newaxis,:],device=device,dtype=torch.int32)
    
    if elmo:
        
        elmo_vocab_file = "vocab-2016-09-10.txt"
        batcher = Batcher(elmo_vocab_file,20)
        
        elmo_in_1 = batcher.batch_sentences([sent1.split()])
        elmo_in_2 = batcher.batch_sentences([sent2.split()])
        
        elmo_in_1 = np.concatenate([elmo_in_1,padding_1],axis=1)
        elmo_in_1 = torch.tensor(elmo_in_1,device=device,dtype=torch.int32)
        
        elmo_in_2 = np.concatenate([elmo_in_2,padding_2],axis=1)
        elmo_in_2 = torch.tensor(elmo_in_2,device=device,dtype=torch.int32)
        
        seq1 = model(inp1,elmo_in_1,test=True)[0]
        p1 = model.pool.score.reshape(-1)[w1_id].detach().cpu().numpy()
        seq2 = model(inp2,elmo_in_2,test=True)[0]
        p2 = model.pool.score.reshape(-1)[w2_id].detach().cpu().numpy()
        
    else:
        
        seq1 = model(inp1,test=True)[0]
        p1 = model.pool.score.reshape(-1)[w1_id].detach().cpu().numpy()
        seq2 = model(inp2,test=True)[0]
        p2 = model.pool.score.reshape(-1)[w2_id].detach().cpu().numpy()
        
    v1 = seq1[w1_id]
    v2 = seq2[w2_id]
    
    sim = torch.cosine_similarity(v1,v2,dim=0).detach().cpu().numpy()
    
    p1,p2,sim = round(float(p1),4),round(float(p2),4),round(float(sim),4)
    
    print(f"'{word}'")
    
    print(f"probability score in '{sent1}' : {p1}")
    print(f"probability score in '{sent2}' : {p2}")
    
    print(f"cosine similarity : {sim}")
    

In [16]:
from sklearn.metrics import classification_report,f1_score,confusion_matrix,recall_score,accuracy_score
import seaborn as sb


def run_metrics(model,data,n_classes):
    
    test,ytest = data

    if n_classes == 1:

        preds = []

        for x,_ in test:
            pred = model(x)
            pred = torch.sigmoid(pred)
            pred = torch.round(pred).reshape(-1).detach().cpu().numpy()
            preds.extend(pred)

        print("f1 score :",round(f1_score(ytest,preds),4))
        print("recall score :",round(recall_score(ytest,preds),4))
        print("accuracy score :",round(accuracy_score(ytest,preds),4))

    elif n_classes > 1:

        preds = []

        for x,_ in test:
            pred = model(x)
            pred = torch.softmax(pred,-1)
            pred = torch.argmax(pred,1).cpu().numpy()
            preds.extend(pred)
        print("f1 score :",round(f1_score(ytest,preds,average='macro'),4))
        print("recall score :",round(recall_score(ytest,preds,average='macro'),4))
        print("accuracy score :",round(accuracy_score(ytest,preds),4))

In [45]:
df = pd.read_csv("/mnt/38DEEB97DEEB4C26/dataset/IMDB Dataset.csv")

X = df.review.apply(lambda x: preprocess_text(x)).values
df.sentiment = df.sentiment.map({'positive':1,'negative':0})
Y = df.sentiment.values

xtrain,xtest,ytrain,ytest = train_test_split(X,Y,train_size=0.7,random_state=0)
xtest,xval,ytest,yval = train_test_split(xtest,ytest,train_size=0.66,random_state=0)

maxlen = min(128-2,max([len(x.split()) for x in xtrain]))

tree_bank = TreebankWordTokenizer()

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xtrain]
xtrain = [' '.join(x) for x in tok_x]

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xtest]
xtest = [' '.join(x) for x in tok_x]

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xval]
xval = [' '.join(x) for x in tok_x]

In [18]:
vocab,data = prepare_data((xtrain,ytrain),(xtest,ytest),(xval,yval),tok,elmo=True)

xtrain,elmo_train,ytrain = data['train']
xtest,elmo_test,ytest = data['test']
xval,elmo_val,yval = data['val']

n_classes = len(np.unique(ytrain))

if n_classes == 2:
    n_classes = n_classes - 1

In [19]:
from torch.utils.data import TensorDataset, DataLoader


BATCH = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def create_elmo_tensor_data(x,y,z,batch,shuffle=True):
    x_ = torch.tensor(x,dtype=torch.int32,device=device)
    y_ = torch.tensor(y,dtype=torch.int32,device=device)
    z_ = torch.tensor(z,dtype=torch.int32,device=device)
    data = TensorDataset(x_,z_,y_)
    data = DataLoader(data,batch_size=batch,shuffle=shuffle)
    return data

def create_tensor_data(x,y,batch,shuffle=True):
    x_ = torch.tensor(x,dtype=torch.int32,device=device)
    y_ = torch.tensor(y,dtype=torch.int32,device=device)
    data = TensorDataset(x_,y_)
    data = DataLoader(data,batch_size=batch,shuffle=shuffle)
    return data


train = create_elmo_tensor_data(xtrain,ytrain,elmo_train,BATCH)
test = create_elmo_tensor_data(xtest,ytest,elmo_test,BATCH * 2,shuffle=False)
valid = create_elmo_tensor_data(xval,yval,elmo_val,BATCH * 2,shuffle=False)

In [20]:
glove_emb = initialize_embeddings(vocab)

In [23]:
reset_random_seeds()

model = ELMoClassifier(vocab,glove_emb)

model = model.to(device)

opt = torch.optim.Adam(model.parameters())

if n_classes > 1:
    loss_fn = nn.CrossEntropyLoss()

else:
    loss_fn = nn.BCEWithLogitsLoss()

lr_scheduler = np.linspace(1e-3,1e-4,len(train)+1)

model

ELMoClassifier(
  (emb): Embedding(10000, 300)
  (rnn): GRU(556, 128, batch_first=True)
  (out): Linear(in_features=128, out_features=1, bias=True)
  (elmo): ELMoLayer(
    (char_cnn): CharCNN(
      (char_emb): Embedding(262, 16)
      (cnn_layers): ModuleList(
        (0): Conv1d(16, 32, kernel_size=(1,), stride=(1,))
        (1): Conv1d(16, 32, kernel_size=(2,), stride=(1,))
        (2): Conv1d(16, 64, kernel_size=(3,), stride=(1,))
        (3): Conv1d(16, 128, kernel_size=(4,), stride=(1,))
        (4): Conv1d(16, 256, kernel_size=(5,), stride=(1,))
        (5): Conv1d(16, 512, kernel_size=(6,), stride=(1,))
        (6): Conv1d(16, 1024, kernel_size=(7,), stride=(1,))
      )
      (gate): Linear(in_features=2048, out_features=2048, bias=True)
      (lin_trans): Linear(in_features=2048, out_features=2048, bias=True)
      (out): Linear(in_features=2048, out_features=128, bias=True)
    )
    (lstm_elmo): LSTMELMo(
      (lstm0): LSTM(128, 1024, proj_size=128, batch_first=True, bidi

In [22]:
train_elmo_model((train,valid),model,opt,loss_fn,epochs=7)

1/7


  0%|          | 0/1094 [00:00<?, ?it/s]

Train Loss = 0.4608
Val Loss = 0.212
--weights saved

2/7


  0%|          | 0/1094 [00:00<?, ?it/s]

Train Loss = 0.3925
Val Loss = 0.2044
--weights saved

3/7


...

Train Loss = 0.335
Val Loss = 0.1878
--weights saved

6/7


  0%|          | 0/1094 [00:00<?, ?it/s]

Train Loss = 0.3189
Val Loss = 0.1862
--weights saved

7/7


  0%|          | 0/1094 [00:00<?, ?it/s]

Train Loss = 0.304
Val Loss = 0.1878



In [53]:
state_dict = torch.load("saved_weights.pt", map_location='cuda')
model.load_state_dict(state_dict)

<All keys matched successfully>


## GRU with ELMo

In [46]:

polysemy_test(word_1,(sents[0],sents[1]),model,elmo=True)


'banks'
probability score in 'the movie banks good money' : 0.067
probability score in 'the movie starts by the river banks' : 0.1521
cosine similarity : 0.85


In [51]:
polysemy_test(word_2,(sents[2],sents[3]),model,elmo=True)

'shot'
probability score in 'the movie was shot well' : 0.2354
probability score in 'he was shot in the movie' : 0.1665
cosine similarity : 0.8402


In [55]:
polysemy_test(word_3,(sents[4],sents[5]),model,elmo=True)

'film'
probability score in 'the film was a hit !' : 0.0151
probability score in 'it was an amazing film' : 0.7604
cosine similarity : 0.4698


In [30]:
reset_random_seeds()

glove_emb = initialize_embeddings(vocab)

model = AttnClassifier(vocab,glove_emb)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

vocab,data = prepare_data((xtrain,ytrain),(xtest,ytest),(xval,yval),tok)

xtrain,ytrain = data['train']
xtest,ytest = data['test']
xval,yval = data['val']

train = create_tensor_data(xtrain,ytrain,BATCH)
test = create_tensor_data(xtest,ytest,BATCH * 2,shuffle=False)
valid = create_tensor_data(xval,yval,BATCH * 2,shuffle=False)

opt = torch.optim.Adam(model.parameters())

if n_classes > 1:
    loss_fn = nn.CrossEntropyLoss()

else:
    loss_fn = nn.BCEWithLogitsLoss()

lr_scheduler = np.linspace(1e-3,1e-4,len(train)+1)

model

AttnClassifier(
  (emb): Embedding(10000, 300)
  (rnn): GRU(300, 128, batch_first=True)
  (out): Linear(in_features=128, out_features=1, bias=True)
  (pool): AttentionPool(
    (w): Linear(in_features=128, out_features=1, bias=False)
  )
  (attn): SelfAttention(
    (q): Linear(in_features=128, out_features=128, bias=False)
    (k): Linear(in_features=128, out_features=128, bias=False)
    (v): Linear(in_features=128, out_features=128, bias=False)
    (attn_drop): Dropout(p=0.1, inplace=False)
    (lin): Linear(in_features=128, out_features=128, bias=True)
    (lin_drop): Dropout(p=0.1, inplace=False)
  )
)

In [31]:
train_model((train,valid),model,opt,loss_fn,epochs=7)

1/7


  0%|          | 0/1094 [00:00<?, ?it/s]

Train Loss = 0.4598
Val Loss = 0.2096
--weights saved

2/7


  0%|          | 0/1094 [00:00<?, ?it/s]

Train Loss = 0.39
Val Loss = 0.1975
--weights saved

3/7


...

Train Loss = 0.3321
Val Loss = 0.1848
--weights saved

6/7


  0%|          | 0/1094 [00:00<?, ?it/s]

Train Loss = 0.3143
Val Loss = 0.1858

7/7


  0%|          | 0/1094 [00:00<?, ?it/s]

Train Loss = 0.2957
Val Loss = 0.1895



In [34]:
state_dict = torch.load("saved_weights.pt", map_location='cuda')
model.load_state_dict(state_dict)

<All keys matched successfully>

## GRU with Self Attention

In [40]:
polysemy_test(word_1,(sents[0],sents[1]),model,elmo=False)

'banks'
probability score in 'the movie banks good money' : 0.1429
probability score in 'the movie starts by the river banks' : 0.1111
cosine similarity : -0.96


In [42]:
polysemy_test(word_2,(sents[2],sents[3]),model,elmo=False)

'shot'
probability score in 'the movie was shot well' : 0.1434
probability score in 'he was shot in the movie' : 0.1257
cosine similarity : -0.646


In [44]:
polysemy_test(word_3,(sents[4],sents[5]),model,elmo=False)

'film'
probability score in 'the film was a hit !' : 0.1252
probability score in 'it was an amazing film' : 0.1273
cosine similarity : 0.989


In [46]:
reset_random_seeds()

model = BaseClassifier(vocab,glove_emb)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

vocab,data = prepare_data((xtrain,ytrain),(xtest,ytest),(xval,yval),tok)

xtrain,ytrain = data['train']
xtest,ytest = data['test']
xval,yval = data['val']

train = create_tensor_data(xtrain,ytrain,BATCH)
test = create_tensor_data(xtest,ytest,BATCH * 2,shuffle=False)
valid = create_tensor_data(xval,yval,BATCH * 2,shuffle=False)

opt = torch.optim.Adam(model.parameters())

if n_classes > 1:
    loss_fn = nn.CrossEntropyLoss()

else:
    loss_fn = nn.BCEWithLogitsLoss()

lr_scheduler = np.linspace(1e-3,1e-4,len(train)+1)

model

BaseClassifier(
  (emb): Embedding(10000, 300)
  (rnn): GRU(300, 128, batch_first=True)
  (out): Linear(in_features=128, out_features=1, bias=True)
  (pool): AttentionPool(
    (w): Linear(in_features=128, out_features=1, bias=False)
  )
)

In [47]:
train_model((train,valid),model,opt,loss_fn,epochs=7)

1/7


  0%|          | 0/1094 [00:00<?, ?it/s]

Train Loss = 0.449
Val Loss = 0.2118
--weights saved

2/7


  0%|          | 0/1094 [00:00<?, ?it/s]

Train Loss = 0.3929
Val Loss = 0.1988
--weights saved

3/7


...

Train Loss = 0.3407
Val Loss = 0.192
--weights saved

6/7


  0%|          | 0/1094 [00:00<?, ?it/s]

Train Loss = 0.3254
Val Loss = 0.1884
--weights saved

7/7


  0%|          | 0/1094 [00:00<?, ?it/s]

Train Loss = 0.3093
Val Loss = 0.1875
--weights saved



In [48]:
state_dict = torch.load("saved_weights.pt", map_location='cuda')
model.load_state_dict(state_dict)

<All keys matched successfully>

## Basic GRU

In [50]:
polysemy_test(word_1,(sents[0],sents[1]),model,elmo=False)

'banks'
probability score in 'the movie banks good money' : 0.1137
probability score in 'the movie starts by the river banks' : 0.057
cosine similarity : 0.7107



In [51]:
polysemy_test(word_2,(sents[2],sents[3]),model,elmo=False)

'shot'
probability score in 'The movie was shot well' : 0.1973
probability score in 'He was shot in the movie' : 0.1754
cosine similarity : 0.7462



In [52]:
polysemy_test(word_3,(sents[4],sents[5]),model,elmo=False)

'film'
probability score in 'The film was a hit!' : 0.0612
probability score in 'It was an amazing film' : 0.6357
cosine similarity : 0.5363

